In [1]:
# ===== Cell 1: 安裝環境（Colab）=====
!pip install -U pip >/dev/null
!pip install "datasets==2.21.0" "transformers==4.44.2" "accelerate==0.34.2" \
             "torchmetrics==1.4.2" "scikit-learn==1.5.2" "pandas==2.2.2" >/dev/null

import sys, platform, torch, transformers, datasets, pandas as pd
print("✅ Python:", platform.python_version())
print("✅ PyTorch:", torch.__version__)
print("✅ Transformers:", transformers.__version__)
print("✅ Datasets:", datasets.__version__)
print("✅ Pandas:", pd.__version__)


✅ Python: 3.12.12
✅ PyTorch: 2.8.0+cu126
✅ Transformers: 4.44.2
✅ Datasets: 2.21.0
✅ Pandas: 2.2.2


In [2]:
# ===== Cell 2: 共用設定與紀錄器 =====
import os, json, time, math, random
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from torchmetrics import PearsonCorrCoef, Accuracy

# reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# 路徑與 run_id
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ROOT = f"/content/outputs/{RUN_ID}"
os.makedirs(ROOT, exist_ok=True)
os.makedirs(f"{ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{ROOT}/preds", exist_ok=True)

# 輔助：logger（會同時 print 並寫入 run.log）
LOG_PATH = f"{ROOT}/run.log"
def log(msg: str):
    stamp = datetime.now().strftime("%H:%M:%S")
    line = f"[{stamp}] {msg}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

# 輔助：CSV 紀錄器（metrics）
METRICS_CSV = f"{ROOT}/metrics.csv"
if not os.path.exists(METRICS_CSV):
    pd.DataFrame(columns=[
        "run_id","model_name","epoch","split",
        "loss_total","loss_reg","loss_cls","pearson","accuracy"
    ]).to_csv(METRICS_CSV, index=False)

def log_metrics(model_name, epoch, split, loss_total=None, loss_reg=None, loss_cls=None, pearson=None, accuracy=None):
    row = {
        "run_id": RUN_ID, "model_name": model_name, "epoch": epoch, "split": split,
        "loss_total": loss_total, "loss_reg": loss_reg, "loss_cls": loss_cls,
        "pearson": pearson, "accuracy": accuracy
    }
    df = pd.DataFrame([row])
    df.to_csv(METRICS_CSV, mode="a", header=False, index=False)

log(f"Outputs folder: {ROOT}")


[05:49:33] Outputs folder: /content/outputs/20251031_054933


In [8]:
# ===== Cell 3.1: Hotfix for dataset column names (entailment_judgment vs judgement) =====
from datasets import DatasetDict

# 檢視實際欄位
train_cols = list(raw["train"].features.keys())
val_cols   = list(raw["validation"].features.keys())
test_cols  = list(raw["test"].features.keys())
log(f"Train columns: {train_cols}")
log(f"Val   columns: {val_cols}")
log(f"Test  columns: {test_cols}")

# 候選鍵（分類 / 回歸）
CLS_CANDIDATES = ["entailment_judgment", "entailment_judgement", "entailment_label", "gold_label", "label"]
REG_CANDIDATES = ["relatedness_score", "relatedness", "score"]

def _pick_key(columns, candidates, name_for_log):
    for k in candidates:
        if k in columns:
            log(f"Detected {name_for_log} column: '{k}'")
            return k
    raise KeyError(f"Cannot find {name_for_log} in columns: {columns}")

CLS_KEY = _pick_key(train_cols, CLS_CANDIDATES, "classification label")
REG_KEY = _pick_key(train_cols, REG_CANDIDATES, "regression label")

# 重新定義 build_dataloaders 使用偵測到的鍵
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

def build_dataloaders(tokenizer_name: str, batch_size=16, max_len=256):
    tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

    def _encode(batch):
        enc = tok(
            batch["premise"], batch["hypothesis"],
            padding="max_length", truncation=True, max_length=max_len
        )
        # 用偵測到的實際欄位名取值
        enc["reg_label"] = batch[REG_KEY]
        enc["cls_label"] = batch[CLS_KEY]
        return enc

    ds_train = raw["train"].map(_encode, batched=True)
    ds_val   = raw["validation"].map(_encode, batched=True)
    ds_test  = raw["test"].map(_encode, batched=True)

    cols = ["input_ids", "attention_mask"]
    # BERT 會有 token_type_ids；RoBERTa 沒有 → 動態相容
    if "token_type_ids" in ds_train.features:
        cols.append("token_type_ids")

    ds_train.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_val.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_test.set_format(type="torch", columns=cols+["reg_label","cls_label"])

    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return tok, dl_train, dl_val, dl_test

log("Hotfix applied: build_dataloaders() now uses auto-detected label keys.")


[05:50:08] Train columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[05:50:08] Val   columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[05:50:08] Test  columns: ['sentence_pair_id', 'premise', 'hypothesis', 'relatedness_score', 'entailment_judgment']
[05:50:08] Detected classification label column: 'entailment_judgment'
[05:50:08] Detected regression label column: 'relatedness_score'
[05:50:08] Hotfix applied: build_dataloaders() now uses auto-detected label keys.


In [9]:
# ===== Cell 3: Load dataset =====
from datasets import load_dataset

raw = load_dataset("SemEvalWorkshop/sem_eval_2014_task_1")
log("Dataset loaded.")

[05:50:12] Dataset loaded.


In [10]:
# ===== Cell 4: 多輸出模型定義（共用） =====
from transformers import AutoModel

class MultiOutputEncoder(nn.Module):
    def __init__(self, encoder_name: str, num_labels_cls: int = 3, dropout: float = 0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        # heads
        self.reg_head = nn.Linear(hidden, 1)            # 回歸：relatedness_score
        self.cls_head = nn.Linear(hidden, num_labels_cls) # 分類：3 類

    def forward(self, batch):
        # 兼容 token_type_ids（RoBERTa 沒有）
        enc_kwargs = {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        }
        if "token_type_ids" in batch:
            enc_kwargs["token_type_ids"] = batch["token_type_ids"]

        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]  # [CLS] 位元（對 RoBERTa 也是第一個 token）
        x = self.dropout(pooled)
        reg = self.reg_head(x).squeeze(-1)    # (B,)
        cls = self.cls_head(x)                # (B, 3)
        return reg, cls


In [11]:
# ===== Cell 5: 訓練/驗證/測試 公用流程 =====
from tqdm.auto import tqdm

def run_train_eval(model_name_key: str,
                   epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256,
                   reg_loss_weight=1.0, cls_loss_weight=1.0, device=None):
    """
    model_name_key: 'bert' 或 'roberta'
    """
    assert model_name_key in ("bert","roberta")
    encoder_name = MODEL_NAMES[model_name_key]
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # Data
    tok, dl_train, dl_val, dl_test = build_dataloaders(encoder_name, batch_size=batch_size, max_len=max_len)

    # Model & Optim
    model = MultiOutputEncoder(encoder_name).to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_reg_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    best_val_score = -1e9
    best_path = f"{ROOT}/checkpoints/{model_name_key}_best.pt"

    log(f"[{model_name_key}] Start training: epochs={epochs}, lr={lr}, bs={batch_size}, max_len={max_len}")
    for ep in range(1, epochs+1):
        model.train()
        tr_loss_total = tr_loss_reg = tr_loss_cls = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_name_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)

            reg_pred, cls_logits = model(batch)
            loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
            loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
            loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()

            tr_loss_total += loss.item()
            tr_loss_reg   += loss_r.item()
            tr_loss_cls   += loss_c.item()

        n_batches = len(dl_train)
        log_metrics(model_name_key, ep, "train",
                    loss_total=tr_loss_total/n_batches,
                    loss_reg=tr_loss_reg/n_batches,
                    loss_cls=tr_loss_cls/n_batches)

        # ---- Validation ----
        model.eval()
        va_loss_total = va_loss_reg = va_loss_cls = 0.0
        pearson_metric = PearsonCorrCoef().to(device)
        acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

        all_reg_pred, all_reg_true = [], []
        all_cls_pred, all_cls_true = [], []

        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_name_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                reg_pred, cls_logits = model(batch)

                loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
                loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
                loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

                va_loss_total += loss.item()
                va_loss_reg   += loss_r.item()
                va_loss_cls   += loss_c.item()

                # metrics
                pearson_metric.update(reg_pred, batch["reg_label"].float())
                acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

                all_reg_pred.extend(reg_pred.detach().cpu().tolist())
                all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
                all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
                all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

        n_val = len(dl_val)
        pear = float(pearson_metric.compute().detach().cpu())
        acc  = float(acc_metric.compute().detach().cpu())
        log_metrics(model_name_key, ep, "val",
                    loss_total=va_loss_total/n_val,
                    loss_reg=va_loss_reg/n_val,
                    loss_cls=va_loss_cls/n_val,
                    pearson=pear, accuracy=acc)
        log(f"[{model_name_key}] ep{ep:02d} | Val Pearson={pear:.4f}, Acc={acc:.4f}")

        # 保存最佳
        score = pear + acc   # 你也可以改成加權：0.5*pear + 0.5*acc
        if score > best_val_score:
            best_val_score = score
            torch.save(model.state_dict(), best_path)
            # 同時保留當前 val 預測
            pd.DataFrame({
                "y_reg_true": all_reg_true,
                "y_reg_pred": all_reg_pred,
                "y_cls_true": all_cls_true,
                "y_cls_pred": all_cls_pred
            }).to_csv(f"{ROOT}/preds/val_{model_name_key}.csv", index=False)
            log(f"[{model_name_key}] ✅ Best updated. Saved to: {best_path}")

    # ---- Test with best checkpoint ----
    log(f"[{model_name_key}] Loading best ckpt: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    pearson_metric = PearsonCorrCoef().to(device)
    acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

    all_reg_pred, all_reg_true = [], []
    all_cls_pred, all_cls_true = [], []

    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_name_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            pearson_metric.update(reg_pred, batch["reg_label"].float())
            acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

            all_reg_pred.extend(reg_pred.detach().cpu().tolist())
            all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
            all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
            all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

    pear = float(pearson_metric.compute().detach().cpu())
    acc  = float(acc_metric.compute().detach().cpu())
    log_metrics(model_name_key, epoch=0, split="test", pearson=pear, accuracy=acc)
    pd.DataFrame({
        "y_reg_true": all_reg_true,
        "y_reg_pred": all_reg_pred,
        "y_cls_true": all_cls_true,
        "y_cls_pred": all_cls_pred
    }).to_csv(f"{ROOT}/preds/test_{model_name_key}.csv", index=False)
    log(f"[{model_name_key}] ✅ Test | Pearson={pear:.4f}, Acc={acc:.4f}")

    return {
        "best_val_score": best_val_score,
        "test_pearson": pear, "test_acc": acc,
        "best_ckpt": best_path
    }


In [15]:
# ===== Cell 6: 跑 BERT-base =====
res_bert = run_train_eval(
    model_name_key="bert",
    epochs=6,           # 正式跑可以調高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[bert] Done. {json.dumps(res_bert, indent=2)}")


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:10<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


...

Valid[bert] ep6:   0%|          | 0/32 [00:00<?, ?it/s]

[06:36:04] [bert] ep06 | Val Pearson=0.8809, Acc=0.8720
[06:36:06] [bert] ✅ Best updated. Saved to: /content/outputs/20251031_054933/checkpoints/bert_best.pt
[06:36:06] [bert] Loading best ckpt: /content/outputs/20251031_054933/checkpoints/bert_best.pt


Test[bert]:   0%|          | 0/308 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[06:37:17] [bert] ✅ Test | Pearson=0.8882, Acc=0.8768
[06:37:17] [bert] Done. {
  "best_val_score": 1.7529194355010986,
  "test_pearson": 0.888213038444519,
  "test_acc": 0.8768013119697571,
  "best_ckpt": "/content/outputs/20251031_054933/checkpoints/bert_best.pt"
}


In [13]:
# ===== Cell 5.5: Define model names dictionary =====
MODEL_NAMES = {
    "bert": "bert-base-uncased",
    "roberta": "roberta-base",
    "gpt2": "gpt2"
}
log("MODEL_NAMES dictionary defined.")

[05:50:39] MODEL_NAMES dictionary defined.


In [14]:
# ===== Cell 7: 跑 RoBERTa-base =====
res_roberta = run_train_eval(
    model_name_key="roberta",
    epochs=6,           # 正式跑可以調高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[roberta] Done. {json.dumps(res_roberta, indent=2)}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

Valid[roberta] ep6:   0%|          | 0/32 [00:00<?, ?it/s]

[06:11:30] [roberta] ep06 | Val Pearson=0.9038, Acc=0.8960
[06:11:34] [roberta] ✅ Best updated. Saved to: /content/outputs/20251031_054933/checkpoints/roberta_best.pt
[06:11:34] [roberta] Loading best ckpt: /content/outputs/20251031_054933/checkpoints/roberta_best.pt


Test[roberta]:   0%|          | 0/308 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 16

[06:12:40] [roberta] ✅ Test | Pearson=0.8983, Acc=0.8926
[06:12:40] [roberta] Done. {
  "best_val_score": 1.7998252511024475,
  "test_pearson": 0.8982547521591187,
  "test_acc": 0.8926324248313904,
  "best_ckpt": "/content/outputs/20251031_054933/checkpoints/roberta_best.pt"
}


In [16]:
# ===== Cell 8: 匯總表（方便貼進報告）=====
df = pd.read_csv(METRICS_CSV)
# 取 test 分數
test_rows = df[(df["split"]=="test") & (df["epoch"]==0)][["model_name","pearson","accuracy"]]
summary = test_rows.copy().reset_index(drop=True)
summary.to_csv(f"{ROOT}/summary.csv", index=False)
log("Saved summary:\n" + summary.to_string(index=False))
summary


[06:39:50] Saved summary:
model_name  pearson  accuracy
   roberta 0.898255  0.892632
      bert 0.888213  0.876801


,model_name,pearson,accuracy
0,roberta,0.898255,0.892632
1,bert,0.888213,0.876801


# GPT-2

In [17]:
# ===== Cell G2-1: Build GPT-2 dataloaders =====
from transformers import AutoTokenizer
from torch.utils.data import DataLoader

def build_dataloaders_gpt2(tokenizer_name: str = "gpt2", batch_size=16, max_len=256):
    tok = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)
    # GPT-2 沒有 pad，統一用 eos 當 pad
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    def _encode(batch):
        # 句對以 eos 當分隔
        texts = [p + tok.eos_token + h for p, h in zip(batch["premise"], batch["hypothesis"])]
        enc = tok(texts, padding="max_length", truncation=True, max_length=max_len, add_special_tokens=True)
        enc["reg_label"] = batch[REG_KEY]
        enc["cls_label"] = batch[CLS_KEY]
        return enc

    ds_train = raw["train"].map(_encode, batched=True)
    ds_val   = raw["validation"].map(_encode, batched=True)
    ds_test  = raw["test"].map(_encode, batched=True)

    cols = ["input_ids", "attention_mask"]  # GPT-2 不使用 token_type_ids
    ds_train.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_val.set_format(type="torch", columns=cols+["reg_label","cls_label"])
    ds_test.set_format(type="torch", columns=cols+["reg_label","cls_label"])

    # 為了避免 colab 的 multiprocessing 噪音，這裡 num_workers=0（穩定）
    dl_train = DataLoader(ds_train, batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=True)
    dl_val   = DataLoader(ds_val,   batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    dl_test  = DataLoader(ds_test,  batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
    return tok, dl_train, dl_val, dl_test

log("GPT-2 dataloader builder ready.")


[06:39:53] GPT-2 dataloader builder ready.


In [18]:
# ===== Cell G2-2: GPT-2 multi-output model =====
import torch
from torch import nn
from transformers import GPT2Model, GPT2Config

class MultiOutputGPT2(nn.Module):
    def __init__(self, encoder_name: str = "gpt2", num_labels_cls: int = 3, dropout: float = 0.1):
        super().__init__()
        self.gpt2 = GPT2Model.from_pretrained(encoder_name)
        # pad_token_id 對齊 tokenizer（上個 cell 已把 pad 設為 eos）
        if self.gpt2.config.pad_token_id is None:
            self.gpt2.config.pad_token_id = self.gpt2.config.eos_token_id
        hidden = self.gpt2.config.n_embd
        self.dropout = nn.Dropout(dropout)
        self.reg_head = nn.Linear(hidden, 1)
        self.cls_head = nn.Linear(hidden, num_labels_cls)

    def forward(self, batch):
        out = self.gpt2(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        last_h = out.last_hidden_state  # (B, L, H)
        # masked mean pooling
        mask = batch["attention_mask"].unsqueeze(-1).float()  # (B, L, 1)
        summed = (last_h * mask).sum(dim=1)                   # (B, H)
        denom = mask.sum(dim=1).clamp(min=1e-6)               # (B, 1)
        pooled = summed / denom
        x = self.dropout(pooled)
        reg = self.reg_head(x).squeeze(-1)   # (B,)
        cls = self.cls_head(x)               # (B,3)
        return reg, cls

log("MultiOutputGPT2 ready.")


[06:39:55] MultiOutputGPT2 ready.


In [19]:
# ===== Cell G2-3: Train & evaluate GPT-2 =====
from torch.optim import AdamW
from torchmetrics import PearsonCorrCoef, Accuracy
from tqdm.auto import tqdm
import pandas as pd
import json

def run_train_eval_gpt2(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256,
                        reg_loss_weight=1.0, cls_loss_weight=1.0, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders_gpt2(batch_size=batch_size, max_len=max_len)

    model = MultiOutputGPT2("gpt2").to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_reg_fn = nn.MSELoss()
    loss_cls_fn = nn.CrossEntropyLoss()

    best_val_score = -1e9
    best_path = f"{ROOT}/checkpoints/gpt2_best.pt"
    model_key = "gpt2"

    log(f"[{model_key}] Start training: epochs={epochs}, lr={lr}, bs={batch_size}, max_len={max_len}")
    for ep in range(1, epochs+1):
        # ---- train ----
        model.train()
        tr_loss_total = tr_loss_reg = tr_loss_cls = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
            loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
            loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            tr_loss_total += loss.item()
            tr_loss_reg   += loss_r.item()
            tr_loss_cls   += loss_c.item()

        n_batches = len(dl_train)
        log_metrics(model_key, ep, "train",
                    loss_total=tr_loss_total/n_batches,
                    loss_reg=tr_loss_reg/n_batches,
                    loss_cls=tr_loss_cls/n_batches)

        # ---- valid ----
        model.eval()
        va_loss_total = va_loss_reg = va_loss_cls = 0.0
        pearson_metric = PearsonCorrCoef().to(device)
        acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

        all_reg_pred, all_reg_true = [], []
        all_cls_pred, all_cls_true = [], []

        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                reg_pred, cls_logits = model(batch)
                loss_r = loss_reg_fn(reg_pred, batch["reg_label"].float())
                loss_c = loss_cls_fn(cls_logits, batch["cls_label"].long())
                loss = reg_loss_weight*loss_r + cls_loss_weight*loss_c

                va_loss_total += loss.item()
                va_loss_reg   += loss_r.item()
                va_loss_cls   += loss_c.item()

                pearson_metric.update(reg_pred, batch["reg_label"].float())
                acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

                all_reg_pred.extend(reg_pred.detach().cpu().tolist())
                all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
                all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
                all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

        pear = float(pearson_metric.compute().detach().cpu())
        acc  = float(acc_metric.compute().detach().cpu())
        log_metrics(model_key, ep, "val",
                    loss_total=va_loss_total/len(dl_val),
                    loss_reg=va_loss_reg/len(dl_val),
                    loss_cls=va_loss_cls/len(dl_val),
                    pearson=pear, accuracy=acc)
        log(f"[{model_key}] ep{ep:02d} | Val Pearson={pear:.4f}, Acc={acc:.4f}")

        score = pear + acc
        if score > best_val_score:
            best_val_score = score
            torch.save(model.state_dict(), best_path)
            pd.DataFrame({
                "y_reg_true": all_reg_true,
                "y_reg_pred": all_reg_pred,
                "y_cls_true": all_cls_true,
                "y_cls_pred": all_cls_pred
            }).to_csv(f"{ROOT}/preds/val_{model_key}.csv", index=False)
            log(f"[{model_key}] ✅ Best updated. Saved to: {best_path}")

    # ---- test ----
    log(f"[{model_key}] Loading best ckpt: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()

    pearson_metric = PearsonCorrCoef().to(device)
    acc_metric = Accuracy(task="multiclass", num_classes=3).to(device)

    all_reg_pred, all_reg_true = [], []
    all_cls_pred, all_cls_true = [], []

    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            reg_pred, cls_logits = model(batch)
            pearson_metric.update(reg_pred, batch["reg_label"].float())
            acc_metric.update(cls_logits.softmax(-1), batch["cls_label"])

            all_reg_pred.extend(reg_pred.detach().cpu().tolist())
            all_reg_true.extend(batch["reg_label"].detach().cpu().tolist())
            all_cls_pred.extend(cls_logits.argmax(-1).detach().cpu().tolist())
            all_cls_true.extend(batch["cls_label"].detach().cpu().tolist())

    pear = float(pearson_metric.compute().detach().cpu())
    acc  = float(acc_metric.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", pearson=pear, accuracy=acc)
    pd.DataFrame({
        "y_reg_true": all_reg_true, "y_reg_pred": all_reg_pred,
        "y_cls_true": all_cls_true, "y_cls_pred": all_cls_pred
    }).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test | Pearson={pear:.4f}, Acc={acc:.4f}")

    return {"best_val_score": best_val_score, "test_pearson": pear, "test_acc": acc, "best_ckpt": best_path}


In [21]:
# ===== Cell G2-4: Run GPT-2 =====
res_gpt2 = run_train_eval_gpt2(
    epochs=6,       # 正式評分可拉高
    lr=2e-5,
    batch_size=16,
    max_len=256
)
log(f"[gpt2] Done. {json.dumps(res_gpt2, indent=2)}")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

[06:40:30] [gpt2] Start training: epochs=6, lr=2e-05, bs=16, max_len=256


Train[gpt2] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[gpt2] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

...

Train[gpt2] ep6:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[gpt2] ep6:   0%|          | 0/32 [00:00<?, ?it/s]

[07:03:40] [gpt2] ep06 | Val Pearson=0.8333, Acc=0.8420
[07:03:50] [gpt2] ✅ Best updated. Saved to: /content/outputs/20251031_054933/checkpoints/gpt2_best.pt
[07:03:50] [gpt2] Loading best ckpt: /content/outputs/20251031_054933/checkpoints/gpt2_best.pt


Test[gpt2]:   0%|          | 0/308 [00:00<?, ?it/s]

[07:05:08] [gpt2] ✅ Test | Pearson=0.8460, Acc=0.8585
[07:05:08] [gpt2] Done. {
  "best_val_score": 1.6753122806549072,
  "test_pearson": 0.8459839224815369,
  "test_acc": 0.8585346341133118,
  "best_ckpt": "/content/outputs/20251031_054933/checkpoints/gpt2_best.pt"
}


In [22]:
# ===== Cell G2-5: Append GPT-2 to summary =====
import pandas as pd
df = pd.read_csv(METRICS_CSV)
test_rows = df[(df["split"]=="test") & (df["epoch"]==0)][["model_name","pearson","accuracy"]]
summary = test_rows.reset_index(drop=True)
summary.to_csv(f"{ROOT}/summary.csv", index=False)
log("Updated summary:\n" + summary.to_string(index=False))
summary


[07:18:27] Updated summary:
model_name  pearson  accuracy
   roberta 0.898255  0.892632
      bert 0.888213  0.876801
      gpt2 0.845984  0.858535


,model_name,pearson,accuracy
0,roberta,0.898255,0.892632
1,bert,0.888213,0.876801
2,gpt2,0.845984,0.858535


# Multi-task

In [23]:
# ===== Cell M1: Single-task BERT models (regression / classification) =====
import torch
from torch import nn
from transformers import AutoModel

class BertRegressor(nn.Module):
    def __init__(self, encoder_name="bert-base-uncased", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, batch):
        enc_kwargs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]}
        if "token_type_ids" in batch: enc_kwargs["token_type_ids"] = batch["token_type_ids"]
        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        reg = self.head(x).squeeze(-1)
        return reg

class BertClassifier(nn.Module):
    def __init__(self, encoder_name="bert-base-uncased", num_labels=3, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, num_labels)

    def forward(self, batch):
        enc_kwargs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"]}
        if "token_type_ids" in batch: enc_kwargs["token_type_ids"] = batch["token_type_ids"]
        out = self.encoder(**enc_kwargs)
        pooled = out.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        logits = self.head(x)
        return logits


In [24]:
# ===== Cell M2: Train/Eval single-task runners (share logs/metrics) =====
from torch.optim import AdamW
from torchmetrics import PearsonCorrCoef, Accuracy
from tqdm.auto import tqdm
import pandas as pd, json, torch

def run_bert_regression(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders("bert-base-uncased", batch_size=batch_size, max_len=max_len)
    model = BertRegressor().to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    model_key = "bert_reg"
    best_val = 1e9
    best_path = f"{ROOT}/checkpoints/{model_key}_best.pt"

    log(f"[{model_key}] start training")
    for ep in range(1, epochs+1):
        model.train(); tr = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            pred = model(batch)
            loss = loss_fn(pred, batch["reg_label"].float())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += loss.item()
        log_metrics(model_key, ep, "train", loss_total=tr/len(dl_train), loss_reg=tr/len(dl_train))

        # valid
        model.eval(); va = 0.0; pear_m = PearsonCorrCoef().to(device)
        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                pred = model(batch)
                loss = loss_fn(pred, batch["reg_label"].float())
                va += loss.item()
                pear_m.update(pred, batch["reg_label"].float())
        pear = float(pear_m.compute().detach().cpu())
        log_metrics(model_key, ep, "val", loss_total=va/len(dl_val), loss_reg=va/len(dl_val), pearson=pear)
        if va < best_val:
            best_val = va
            torch.save(model.state_dict(), best_path)
            log(f"[{model_key}] ✅ best updated -> {best_path}")

    # test
    log(f"[{model_key}] load best: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device)); model.eval()
    pear_m = PearsonCorrCoef().to(device)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            pred = model(batch)
            pear_m.update(pred, batch["reg_label"].float())
            y_true += batch["reg_label"].detach().cpu().tolist()
            y_pred += pred.detach().cpu().tolist()
    pear = float(pear_m.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", pearson=pear)
    pd.DataFrame({"y_reg_true": y_true, "y_reg_pred": y_pred}).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test Pearson={pear:.4f}")
    return {"test_pearson": pear, "best_ckpt": best_path}

def run_bert_classification(epochs=3, lr=2e-5, weight_decay=0.01, batch_size=16, max_len=256, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tok, dl_train, dl_val, dl_test = build_dataloaders("bert-base-uncased", batch_size=batch_size, max_len=max_len)
    model = BertClassifier().to(device)
    opt = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.CrossEntropyLoss()
    model_key = "bert_cls"
    best_val = -1.0
    best_path = f"{ROOT}/checkpoints/{model_key}_best.pt"

    log(f"[{model_key}] start training")
    for ep in range(1, epochs+1):
        model.train(); tr = 0.0
        for batch in tqdm(dl_train, desc=f"Train[{model_key}] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            logits = model(batch)
            loss = loss_fn(logits, batch["cls_label"].long())
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tr += loss.item()
        log_metrics(model_key, ep, "train", loss_total=tr/len(dl_train), loss_cls=tr/len(dl_train))

        # valid
        model.eval(); va = 0.0; acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
        with torch.no_grad():
            for batch in tqdm(dl_val, desc=f"Valid[{model_key}] ep{ep}"):
                for k in batch: batch[k] = batch[k].to(device)
                logits = model(batch)
                loss = loss_fn(logits, batch["cls_label"].long())
                va += loss.item()
                acc_m.update(logits.softmax(-1), batch["cls_label"])
        acc = float(acc_m.compute().detach().cpu())
        log_metrics(model_key, ep, "val", loss_total=va/len(dl_val), loss_cls=va/len(dl_val), accuracy=acc)
        if acc > best_val:
            best_val = acc
            torch.save(model.state_dict(), best_path)
            log(f"[{model_key}] ✅ best updated -> {best_path}")

    # test
    log(f"[{model_key}] load best: {best_path}")
    model.load_state_dict(torch.load(best_path, map_location=device)); model.eval()
    acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in tqdm(dl_test, desc=f"Test[{model_key}]"):
            for k in batch: batch[k] = batch[k].to(device)
            logits = model(batch)
            acc_m.update(logits.softmax(-1), batch["cls_label"])
            y_true += batch["cls_label"].detach().cpu().tolist()
            y_pred += logits.argmax(-1).detach().cpu().tolist()
    acc = float(acc_m.compute().detach().cpu())
    log_metrics(model_key, epoch=0, split="test", accuracy=acc)
    pd.DataFrame({"y_cls_true": y_true, "y_cls_pred": y_pred}).to_csv(f"{ROOT}/preds/test_{model_key}.csv", index=False)
    log(f"[{model_key}] ✅ Test Acc={acc:.4f}")
    return {"test_acc": acc, "best_ckpt": best_path}


In [25]:
# ===== Cell M3: Run single-task baselines =====
res_reg = run_bert_regression(epochs=3, lr=2e-5, batch_size=16, max_len=256)
res_cls = run_bert_classification(epochs=3, lr=2e-5, batch_size=16, max_len=256)
log(f"[bert_reg] {json.dumps(res_reg, indent=2)}")
log(f"[bert_cls] {json.dumps(res_cls, indent=2)}")


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

[07:18:39] [bert_reg] start training


Train[bert_reg] ep1:   0%|          | 0/282 [00:00<?, ?it/s]

Valid[bert_reg] ep1:   0%|          | 0/32 [00:00<?, ?it/s]

...

Valid[bert_cls] ep3:   0%|          | 0/32 [00:00<?, ?it/s]

[07:39:40] [bert_cls] ✅ best updated -> /content/outputs/20251031_054933/checkpoints/bert_cls_best.pt
[07:39:40] [bert_cls] load best: /content/outputs/20251031_054933/checkpoints/bert_cls_best.pt


Test[bert_cls]:   0%|          | 0/308 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1647, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child processException ignored in: 
<function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
Exception ignored in: Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1664, in __del__
    <function _MultiProcessingDataLoaderIter.__del__ at 0x7dc9f40e85e0>
self._shutdown_workers(

[07:40:49] [bert_cls] ✅ Test Acc=0.8731
[07:40:50] [bert_reg] {
  "test_pearson": 0.8723450303077698,
  "best_ckpt": "/content/outputs/20251031_054933/checkpoints/bert_reg_best.pt"
}
[07:40:50] [bert_cls] {
  "test_acc": 0.8731479644775391,
  "best_ckpt": "/content/outputs/20251031_054933/checkpoints/bert_cls_best.pt"
}


In [26]:
# ===== Cell M4: Build comparison summary =====
import pandas as pd

df = pd.read_csv(METRICS_CSV)

want = df[(df["split"]=="test") & (df["epoch"]==0)][
    ["model_name","pearson","accuracy"]
].fillna("")

# 只挑出 bert (multi-output), bert_reg, bert_cls
mask = want["model_name"].isin(["bert","bert_reg","bert_cls"])
summary_bert = want[mask].rename(columns={
    "model_name":"model",
    "pearson":"test_pearson",
    "accuracy":"test_acc"
}).reset_index(drop=True)

summary_bert.to_csv(f"{ROOT}/summary_bert_multi_vs_single.csv", index=False)
log("Saved summary_bert_multi_vs_single.csv")
summary_bert


[07:43:04] Saved summary_bert_multi_vs_single.csv


,model,test_pearson,test_acc
0,bert,0.888213,0.876801
1,bert_reg,0.872345,
2,bert_cls,,0.873148


# Error Analysis

In [29]:
# ===== Cell EA-1: load preds and build analysis table =====
import os, pandas as pd
from datasets import load_dataset

MODEL_FOR_ANALYSIS = "roberta"   # 可改成 "bert" / "gpt2"
TEST_PREDS = f"{ROOT}/preds/test_{MODEL_FOR_ANALYSIS}.csv"
assert os.path.exists(TEST_PREDS), f"找不到 {TEST_PREDS}，請先完成該模型的測試推論。"

# 讀測試預測
preds = pd.read_csv(TEST_PREDS)

# 讀原始資料（保持與前面一致）
raw = load_dataset("SemEvalWorkshop/sem_eval_2014_task_1", trust_remote_code=True)

# 把 datasets 的欄位拉出來做 DataFrame
df_raw = pd.DataFrame({
    "premise": raw["test"]["premise"],
    "hypothesis": raw["test"]["hypothesis"],
    "y_reg_true_raw": raw["test"][REG_KEY], # Rename to avoid duplicate column name after merge
    "y_cls_true_raw": raw["test"][CLS_KEY]  # Rename to avoid duplicate column name after merge
})

# Add a temporary index column for merging
df_raw['__index'] = df_raw.index
preds['__index'] = preds.index

# Merge based on the temporary index
df = pd.merge(df_raw, preds, on='__index').drop(columns=['__index'])

# 派生欄位
df["reg_residual"] = df["y_reg_pred"] - df["y_reg_true_raw"] # Use renamed column
df["reg_abs_err"]  = df["reg_residual"].abs()
df["cls_correct"]  = (df["y_cls_pred"] == df["y_cls_true_raw"]).astype(int) # Use renamed column

# 儲存完整分析表
OUT_CSV = f"{ROOT}/error_analysis_{MODEL_FOR_ANALYSIS}.csv"
df.to_csv(OUT_CSV, index=False)
log(f"[EA] Saved analysis table: {OUT_CSV}")
display(df.head(3))

[07:43:58] [EA] Saved analysis table: /content/outputs/20251031_054933/error_analysis_roberta.csv


,premise,hypothesis,y_reg_true_raw,y_cls_true_raw,y_reg_true,y_reg_pred,y_cls_true,y_cls_pred,reg_residual,reg_abs_err,cls_correct
0,There is no boy playing outdoors and there is ...,A group of kids is playing in a yard and an ol...,3.3,0,3.3,2.844054,0,0,-0.455946,0.455946,1
1,A group of boys in a yard is playing and a man...,The young boys are playing outdoors and the ma...,3.7,0,3.7,4.180803,0,0,0.480803,0.480803,1
2,A group of children is playing in the house an...,The young boys are playing outdoors and the ma...,3.0,0,3.0,3.199329,0,0,0.199329,0.199329,1


In [30]:
# ===== Cell EA-2: classification error analysis =====
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

CLASS_NAMES = ["NEUTRAL(0)", "ENTAIL(1)", "CONTRA(2)"]

cm = confusion_matrix(df["y_cls_true"], df["y_cls_pred"], labels=[0,1,2])
report = classification_report(df["y_cls_true"], df["y_cls_pred"], labels=[0,1,2], target_names=CLASS_NAMES, digits=4)
log("[EA] Classification report:\n" + report)

# 畫混淆矩陣
plt.figure(figsize=(4.5,4))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Confusion Matrix — {MODEL_FOR_ANALYSIS}")
plt.xticks([0,1,2], CLASS_NAMES, rotation=40, ha="right")
plt.yticks([0,1,2], CLASS_NAMES)
for i in range(3):
    for j in range(3):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.tight_layout()
FIG_CM = f"{ROOT}/fig_confusion_{MODEL_FOR_ANALYSIS}.png"
plt.savefig(FIG_CM, dpi=140); plt.close()
log(f"[EA] Saved confusion matrix: {FIG_CM}")

# 最常見的誤判對（例如 NEUTRAL<->ENTAIL）
pair_counts = (
    df.loc[df["cls_correct"]==0, ["y_cls_true","y_cls_pred"]]
      .value_counts()
      .rename("n").reset_index()
      .sort_values("n", ascending=False)
)
TOP_PAIR_CSV = f"{ROOT}/cls_top_confusions_{MODEL_FOR_ANALYSIS}.csv"
pair_counts.to_csv(TOP_PAIR_CSV, index=False)
log(f"[EA] Saved top confused pairs: {TOP_PAIR_CSV}")

# 抽出最常見的誤判類型的 Top-10 例句
if not pair_counts.empty:
    t, p = int(pair_counts.iloc[0]["y_cls_true"]), int(pair_counts.iloc[0]["y_cls_pred"])
    top_confused = df[(df["y_cls_true"]==t) & (df["y_cls_pred"]==p)].copy()
    top_confused = top_confused.assign(
        err_tag=lambda d: "TRUE="+CLASS_NAMES[t]+" | PRED="+CLASS_NAMES[p]
    ).head(10)
    TOP_EX_CSV = f"{ROOT}/cls_top_examples_{MODEL_FOR_ANALYSIS}.csv"
    top_confused[["err_tag","premise","hypothesis","y_cls_true","y_cls_pred","y_reg_true","y_reg_pred"]].to_csv(TOP_EX_CSV, index=False)
    log(f"[EA] Saved sample confused examples: {TOP_EX_CSV}")


[07:45:21] [EA] Classification report:
              precision    recall  f1-score   support

  NEUTRAL(0)     0.9084    0.9051    0.9067      2793
   ENTAIL(1)     0.8486    0.8918    0.8697      1414
   CONTRA(2)     0.9255    0.8458    0.8839       720

    accuracy                         0.8926      4927
   macro avg     0.8942    0.8809    0.8868      4927
weighted avg     0.8937    0.8926    0.8928      4927

[07:45:21] [EA] Saved confusion matrix: /content/outputs/20251031_054933/fig_confusion_roberta.png
[07:45:21] [EA] Saved top confused pairs: /content/outputs/20251031_054933/cls_top_confusions_roberta.csv
[07:45:21] [EA] Saved sample confused examples: /content/outputs/20251031_054933/cls_top_examples_roberta.csv


In [31]:
# ===== Cell EA-3: regression error analysis =====
import numpy as np, matplotlib.pyplot as plt

# 指標：RMSE/MAE
rmse = float(np.sqrt(np.mean((df["y_reg_pred"] - df["y_reg_true"])**2)))
mae  = float(np.mean(np.abs(df["y_reg_pred"] - df["y_reg_true"])))
log(f"[EA] Regression RMSE={rmse:.4f}, MAE={mae:.4f}")

# 真實 vs 預測
plt.figure(figsize=(4.6,4))
plt.scatter(df["y_reg_true"], df["y_reg_pred"], s=6)
plt.plot([1,5],[1,5], linestyle="--")
plt.xlabel("True relatedness"); plt.ylabel("Pred relatedness")
plt.title(f"True vs Pred — {MODEL_FOR_ANALYSIS}")
plt.tight_layout()
FIG_SCAT = f"{ROOT}/fig_scatter_reg_{MODEL_FOR_ANALYSIS}.png"
plt.savefig(FIG_SCAT, dpi=140); plt.close()
log(f"[EA] Saved regression scatter: {FIG_SCAT}")

# 殘差直方圖
plt.figure(figsize=(4.6,4))
plt.hist(df["reg_residual"], bins=40)
plt.axvline(0, color="k", linestyle="--")
plt.title(f"Residuals (pred-true) — {MODEL_FOR_ANALYSIS}")
plt.tight_layout()
FIG_RES = f"{ROOT}/fig_residuals_{MODEL_FOR_ANALYSIS}.png"
plt.savefig(FIG_RES, dpi=140); plt.close()
log(f"[EA] Saved residual histogram: {FIG_RES}")

# Top-20 最大誤差樣本
topk = df.sort_values("reg_abs_err", ascending=False).head(20)
TOPK_CSV = f"{ROOT}/reg_top_abs_errors_{MODEL_FOR_ANALYSIS}.csv"
topk[["premise","hypothesis","y_reg_true","y_reg_pred","reg_abs_err","y_cls_true","y_cls_pred"]].to_csv(TOPK_CSV, index=False)
log(f"[EA] Saved top-20 regression errors: {TOPK_CSV}")


[07:45:25] [EA] Regression RMSE=0.5018, MAE=0.3869
[07:45:25] [EA] Saved regression scatter: /content/outputs/20251031_054933/fig_scatter_reg_roberta.png
[07:45:25] [EA] Saved residual histogram: /content/outputs/20251031_054933/fig_residuals_roberta.png
[07:45:25] [EA] Saved top-20 regression errors: /content/outputs/20251031_054933/reg_top_abs_errors_roberta.csv


In [32]:
# ===== Cell EA-4: overlap of hard cases =====
HARD_K = 200  # 取回歸誤差最大的前 K 筆
hard_reg_idx = set(df.sort_values("reg_abs_err", ascending=False).head(HARD_K).index.tolist())
hard_cls_idx = set(df[df["cls_correct"]==0].index.tolist())

overlap_idx = sorted(list(hard_reg_idx & hard_cls_idx))
overlap = df.loc[overlap_idx, ["premise","hypothesis","y_reg_true","y_reg_pred","reg_abs_err","y_cls_true","y_cls_pred"]]

OVERLAP_CSV = f"{ROOT}/hard_overlap_{MODEL_FOR_ANALYSIS}.csv"
overlap.to_csv(OVERLAP_CSV, index=False)
ratio = (len(overlap_idx) / max(1, len(hard_cls_idx))) * 100
log(f"[EA] Hard-overlap saved ({len(overlap_idx)} samples, {ratio:.1f}% of misclassified). Path: {OVERLAP_CSV}")
overlap.head(3)


[07:45:28] [EA] Hard-overlap saved (94 samples, 17.8% of misclassified). Path: /content/outputs/20251031_054933/hard_overlap_roberta.csv


,premise,hypothesis,y_reg_true,y_reg_pred,reg_abs_err,y_cls_true,y_cls_pred
11,A person in a black jacket is doing tricks on ...,A person on a black motorbike is doing tricks ...,3.0,4.683064,1.683064,0,1
25,Two spectators are kickboxing and some people ...,Two people are kickboxing and spectators are w...,4.0,5.109640,1.109640,0,1
46,A little girl is looking at a woman in costume,A little girl in costume looks like a woman,2.9,4.370373,1.470373,0,1


In [ ]:
# ===== Cell EA-5A: Temperature scaling on validation to calibrate classification =====
import torch
import torch.nn as nn
import numpy as np
from torch.optim import LBFGS
from tqdm.auto import tqdm

# 先在「驗證集」上擬合溫度
_, dl_train, dl_val, dl_test = build_dataloaders(MODEL_NAMES.get(MODEL_FOR_ANALYSIS, "roberta-base"))
from transformers import AutoModel

# 取出已訓練好的多輸出模型
if MODEL_FOR_ANALYSIS in ("bert","roberta"):
    enc_name = MODEL_NAMES[MODEL_FOR_ANALYSIS]
    model = MultiOutputEncoder(enc_name).to("cpu")
    best_path = f"{ROOT}/checkpoints/{MODEL_FOR_ANALYSIS}_best.pt"
elif MODEL_FOR_ANALYSIS=="gpt2":
    model = MultiOutputGPT2("gpt2").to("cpu")
    best_path = f"{ROOT}/checkpoints/gpt2_best.pt"
else:
    raise ValueError("MODEL_FOR_ANALYSIS not supported")

model.load_state_dict(torch.load(best_path, map_location="cpu"))
model.eval()

# 收集 val logits/labels
all_logits, all_y = [], []
with torch.no_grad():
    for batch in tqdm(dl_val, desc="Collect val logits"):
        reg_pred, cls_logits = model({k:v for k,v in batch.items()})
        all_logits.append(cls_logits)
        all_y.append(batch["cls_label"])
logits_val = torch.cat(all_logits, dim=0)
y_val = torch.cat(all_y, dim=0)

# 溫度參數
temperature = nn.Parameter(torch.ones(1))

def nll_with_temp():
    scaled = logits_val / temperature.clamp(min=1e-3)
    return nn.CrossEntropyLoss()(scaled, y_val)

opt = LBFGS([temperature], lr=0.1, max_iter=50)

def closure():
    opt.zero_grad()
    loss = nll_with_temp()
    loss.backward()
    return loss

opt.step(closure)
T = float(temperature.data)
log(f"[EA] Fitted temperature on val: T={T:.3f}")

# 用校正後溫度重新在 test 上計算 Acc
from torchmetrics import Accuracy
acc_metric = Accuracy(task="multiclass", num_classes=3)
with torch.no_grad():
    for batch in tqdm(dl_test, desc="Re-eval with temperature"):
        reg_pred, cls_logits = model({k:v for k,v in batch.items()})
        scaled = cls_logits / max(T, 1e-3)
        acc_metric.update(scaled.softmax(-1), batch["cls_label"])
acc_cal = float(acc_metric.compute())
log(f"[EA] Test accuracy after temperature scaling: {acc_cal:.4f}")


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Collect val logits:   0%|          | 0/32 [00:00<?, ?it/s]

[07:49:48] [EA] Fitted temperature on val: T=1.584


/usr/local/lib/python3.12/dist-packages/torch/optim/lbfgs.py:457: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  loss = float(closure())


Re-eval with temperature:   0%|          | 0/308 [00:00<?, ?it/s]

In [ ]:
# ===== Cell EA-5B: Swap MSE -> SmoothL1 for regression and re-train quickly (few epochs) =====
# 輕量嘗試：以相同超參、把 loss_reg_fn 改成 SmoothL1Loss 訓練 1~2 個 epoch，看 RMSE/MAE 是否下降
def run_train_eval_smoothL1(model_name_key="roberta", epochs=2, lr=2e-5, batch_size=16, max_len=256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    encoder_name = MODEL_NAMES[model_name_key] if model_name_key in MODEL_NAMES else "roberta-base"
    tok, dl_train, dl_val, dl_test = build_dataloaders(encoder_name, batch_size=batch_size, max_len=max_len)
    model = MultiOutputEncoder(encoder_name).to(device)
    opt = AdamW(model.parameters(), lr=lr)
    loss_reg_fn = nn.SmoothL1Loss(beta=0.5)
    loss_cls_fn = nn.CrossEntropyLoss()

    best = -1e9
    for ep in range(1, epochs+1):
        model.train()
        for batch in tqdm(dl_train, desc=f"Train[SmoothL1] ep{ep}"):
            for k in batch: batch[k] = batch[k].to(device)
            reg, cls = model(batch)
            loss = loss_reg_fn(reg, batch["reg_label"].float()) + loss_cls_fn(cls, batch["cls_label"].long())
            opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()

        # val
        model.eval()
        pear_m = PearsonCorrCoef().to(device); acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
        with torch.no_grad():
            for batch in dl_val:
                for k in batch: batch[k] = batch[k].to(device)
                reg, cls = model(batch)
                pear_m.update(reg, batch["reg_label"].float())
                acc_m.update(cls.softmax(-1), batch["cls_label"])
        score = float(pear_m.compute()+acc_m.compute())
        if score > best:
            best = score
            torch.save(model.state_dict(), f"{ROOT}/checkpoints/{model_name_key}_smoothL1_best.pt")
            log("[EA] SmoothL1 best updated")

    # test
    model.load_state_dict(torch.load(f"{ROOT}/checkpoints/{model_name_key}_smoothL1_best.pt", map_location=device))
    model.eval()
    pear_m = PearsonCorrCoef().to(device); acc_m = Accuracy(task="multiclass", num_classes=3).to(device)
    with torch.no_grad():
        for batch in dl_test:
            for k in batch: batch[k] = batch[k].to(device)
            reg, cls = model(batch)
            pear_m.update(reg, batch["reg_label"].float()); acc_m.update(cls.softmax(-1), batch["cls_label"])
    pear, acc = float(pear_m.compute()), float(acc_m.compute())
    log(f"[EA] SmoothL1 Test | Pearson={pear:.4f}, Acc={acc:.4f}")
    return pear, acc

run_train_eval_smoothL1("roberta", epochs=2)
